In [1]:
!pip install -U langchain langchain-google-genai langchain-chroma chromadb

In [2]:
import os
from datetime import datetime

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage

In [3]:
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)

In [6]:
# =========================
# 1. API KEY
# =========================

#os.environ["GOOGLE_API_KEY"] = "YOUR_GEMINI_API_KEY"

# =========================
# 2. LLM
# =========================

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    api_key=api_key
)

# =========================
# 3. VECTOR DATABASE - CHROMADB
# =========================

docs = [
    Document(page_content="Machine Learning is a branch of AI that learns patterns from data."),
    Document(page_content="Deep Learning uses neural networks with many layers."),
    Document(page_content="RAG means Retrieval Augmented Generation. It uses vector databases to retrieve relevant knowledge."),
    Document(page_content="LangChain is used to build LLM applications with prompts, chains, agents, tools, memory and retrieval."),
    Document(page_content="Agentic AI means AI systems that can plan, use tools, remember context, and perform tasks.")
]

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    api_key=api_key
)

vector_db = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="knowledge_base",
    persist_directory="./chroma_multi_agent_db"
)

retriever = vector_db.as_retriever(search_kwargs={"k": 2})

In [7]:
# =========================
# 4. TOOLS
# =========================

@tool
def get_current_time() -> str:
    """Get the current date and time."""
    return str(datetime.now())


@tool
def calculator(expression: str) -> str:
    """Calculate a simple mathematical expression. Example: 20*5 + 10"""
    try:
        allowed_chars = "0123456789+-*/(). "
        if not all(ch in allowed_chars for ch in expression):
            return "Invalid expression. Only basic arithmetic is allowed."

        result = eval(expression)
        return str(result)

    except Exception as e:
        return f"Calculation error: {e}"


@tool
def search_knowledge_base(query: str) -> str:
    """Search the ChromaDB vector database for useful knowledge."""
    results = retriever.invoke(query)

    if not results:
        return "No relevant knowledge found."

    return "\n".join([doc.page_content for doc in results])


@tool
def save_output_to_file(content: str) -> str:
    """Save final output to a text file."""
    file_name = "agent_output.txt"

    with open(file_name, "w", encoding="utf-8") as f:
        f.write(content)

    return f"Output saved to {file_name}"


tools = [
    get_current_time,
    calculator,
    search_knowledge_base,
    save_output_to_file
]

In [8]:
# =========================
# 5. MULTIPLE AGENTS
# =========================

research_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a Research Agent.
    Your job is to search knowledge, explain concepts, and answer factual questions.
    Use search_knowledge_base when needed.
    Answer in simple English.
    """
)

writing_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a Writing Agent.
    Your job is to write blogs, summaries, emails, notes, and explanations.
    Use simple English.
    Use save_output_to_file if the user asks to save.
    """
)

math_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a Math Agent.
    Your job is to solve arithmetic and calculation-related questions.
    Use calculator tool whenever calculation is needed.
    Show the final answer clearly.
    """
)

general_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a General Assistant Agent.
    Answer general questions clearly and simply.
    Use tools when useful.
    """
)

In [9]:
# =========================
# 6. MULTI-USER MEMORY
# =========================

user_memory = {
    "user1": [],
    "user2": [],
    "user3": []
}

# =========================
# 7. ROUTER AGENT FUNCTION
# =========================

def choose_agent(task: str):
    task_lower = task.lower()

    if any(word in task_lower for word in ["calculate", "sum", "multiply", "divide", "+", "-", "*", "/"]):
        return math_agent, "Math Agent"

    elif any(word in task_lower for word in ["write", "blog", "email", "summary", "article", "notes"]):
        return writing_agent, "Writing Agent"

    elif any(word in task_lower for word in ["what is", "explain", "research", "rag", "langchain", "machine learning", "agentic"]):
        return research_agent, "Research Agent"

    else:
        return general_agent, "General Agent"


In [10]:
# =========================
# 8. MAIN MULTI-USER CHAT FUNCTION
# =========================

def ask_multi_agent(user_id: str, user_input: str):
    if user_id not in user_memory:
        user_memory[user_id] = []

    selected_agent, agent_name = choose_agent(user_input)

    memory_text = ""

    for msg in user_memory[user_id]:
        if isinstance(msg, HumanMessage):
            memory_text += f"User: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            memory_text += f"Assistant: {msg.content}\n"

    final_input = f"""
    Previous conversation with this user:
    {memory_text}

    Current user request:
    {user_input}
    """

    response = selected_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": final_input
            }
        ]
    })

    answer = response["messages"][-1].content

    user_memory[user_id].append(HumanMessage(content=user_input))
    user_memory[user_id].append(AIMessage(content=answer))

    return agent_name, answer


In [11]:

# =========================
# 9. INTERACTIVE MULTI-USER LOOP
# =========================

print("Multi-User Multi-Agent System Started")
print("Users: user1, user2, user3")
print("Type 'exit' to stop\n")

while True:
    user_id = input("Enter user id: ")

    if user_id.lower() == "exit":
        break

    user_question = input("Enter task/question: ")

    if user_question.lower() == "exit":
        break

    agent_used, answer = ask_multi_agent(user_id, user_question)

    print("\nAgent Used:", agent_used)
    print("Answer:\n", answer)
    print("-" * 80)

Multi-User Multi-Agent System Started
Users: user1, user2, user3
Type 'exit' to stop

Enter user id: user1
Enter task/question: what is rag?

Agent Used: Research Agent
Answer:
 [{'type': 'text', 'text': "RAG stands for Retrieval Augmented Generation. It's a method that uses vector databases to find and retrieve relevant information.", 'extras': {'signature': 'Cq8CAQw51sd7CHIqVkIvnXN4d/f2c6PGcl5FHWj3ZRN6J6QLqFW719/Ph8d6xR92X1jaCxD6PmQPzXqAK+0dIkzVaaKvWh9yI/+xx1GPvu2pl2ysO2+ZaRkK72H6WNOz1fj614zIHDlaVK1LYESdZzjLhMwZtM0qz9Ym2+494WjfutQ5Lcz1NeJksImDd3lmjsSmeKiYHiyts1rh9k3HK6G8IKab1Qt2VPCgTukfN86u/FkR3MpXNzUreeYj70aSJJywoKSO2/S/53GmXohj0yj/FfmHJqvTTVKLN+Tpsh5uoLmljseNQERaIqblTmP+eDOVAr/proD+vpdGe69cpIEosaxCxqbYd0+l9NgTkxFLTQ07H9bxTFBparuVu3S3F+mOsNYbMPesdDIgyJ4+owGn'}}]
--------------------------------------------------------------------------------
Enter user id: user2
Enter task/question: who is Lin Dan?

Agent Used: General Agent
Answer:
 [{'type': 'text', 'text': 'I am sorry, I cannot f